In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from Pipeline.Model.EvaluationELM import EvaluationELM
from Pipeline.Model.EvaluationMatrix import EvaluationMatrix
from Pipeline.Model.ExtremeLearningMachine import ExtremeLearningMachine

In [2]:
def extract_optimal_vector(dataframe: pd.DataFrame,
                           metric_col: str = 'Clinical_F2_LCB') -> pd.Series:
    return dataframe.dropna(subset=[metric_col]).sort_values(by=metric_col, ascending=False).iloc[1]

In [3]:
filePath = '../../Dataset/UCI_Gallstone_Dataset.csv'
df = pd.read_csv(filePath)

targetCol = ['Gallstone Status']
X = df.drop(targetCol, axis=1)
y = df[targetCol]
features_size = X.shape[1]

In [4]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [5]:
hidden_size_explore = range(40, 61)
lambda_value_explore = 2.0 ** np.arange(-10, 11)

In [6]:
# 1. Instantiate the Evaluator Object ONCE
evaluator = EvaluationELM(
    x=X,
    y=y,
    activation_function=sigmoid,
    elm_initial_state_range=range(41, 50),
    data_split_state_range=range(1, 6),
    test_size=0.2,
    k_fold=5
)

In [7]:
hidden_size_record ,hidden_size_result = evaluator.grid_search_hidden_size(hidden_size_explore)

In [8]:
best_hidden_size = extract_optimal_vector(hidden_size_result)
best_hidden_size_value = best_hidden_size['Hidden_Nodes']
best_hidden_size

Hidden_Nodes                            50
Activation                         sigmoid
Lambda_Value                           0.0
avg_Accuracy_Seed_Mean              0.6945
avg_Accuracy_Seed_Std               0.0591
avg_Accuracy_Seed_Min               0.5098
avg_Accuracy_Seed_Max               0.8431
avg_Precision_Seed_Mean             0.7064
avg_Precision_Seed_Std              0.0709
avg_Precision_Seed_Min                 0.5
avg_Precision_Seed_Max              0.9444
avg_Recall_Seed_Mean                0.6625
avg_Recall_Seed_Std                 0.0929
avg_Recall_Seed_Min                    0.4
avg_Recall_Seed_Max                   0.88
avg_NPV_Seed_Mean                   0.6908
avg_NPV_Seed_Std                    0.0613
avg_NPV_Seed_Min                    0.5161
avg_NPV_Seed_Max                    0.8696
avg_Specificity_Seed_Mean           0.7256
avg_Specificity_Seed_Std            0.0904
avg_Specificity_Seed_Min            0.4231
avg_Specificity_Seed_Max            0.9615
avg_F1-Scor

In [9]:
lambda_record, lambda_result = evaluator.grid_search_lambda(
    hidden_size=best_hidden_size_value,
    lambda_range=lambda_value_explore
)

In [10]:
best_lambda_value = extract_optimal_vector(lambda_result)
best_lambda_value

Hidden_Nodes                             50
Activation                          sigmoid
Lambda_Value                       1.584893
avg_Accuracy_Seed_Mean               0.7093
avg_Accuracy_Seed_Std                0.0598
avg_Accuracy_Seed_Min                 0.549
avg_Accuracy_Seed_Max                0.8824
avg_Precision_Seed_Mean              0.7265
avg_Precision_Seed_Std               0.0745
avg_Precision_Seed_Min                 0.55
avg_Precision_Seed_Max               0.9545
avg_Recall_Seed_Mean                 0.6703
avg_Recall_Seed_Std                  0.0933
avg_Recall_Seed_Min                    0.44
avg_Recall_Seed_Max                    0.88
avg_NPV_Seed_Mean                    0.7022
avg_NPV_Seed_Std                     0.0619
avg_NPV_Seed_Min                     0.5484
avg_NPV_Seed_Max                     0.8696
avg_Specificity_Seed_Mean            0.7473
avg_Specificity_Seed_Std             0.0913
avg_Specificity_Seed_Min             0.4615
avg_Specificity_Seed_Max        

In [11]:
combination_record , combination_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_explore,
    lambda_range=lambda_value_explore
)
best_combination_result = extract_optimal_vector(combination_result)
best_combination_result

Hidden_Nodes                              59
Activation                           sigmoid
Lambda_Value                       15.848932
avg_Accuracy_Seed_Mean                0.7336
avg_Accuracy_Seed_Std                 0.0536
avg_Accuracy_Seed_Min                  0.549
avg_Accuracy_Seed_Max                 0.8627
avg_Precision_Seed_Mean               0.7652
avg_Precision_Seed_Std                0.0762
avg_Precision_Seed_Min                0.5455
avg_Precision_Seed_Max                0.9444
avg_Recall_Seed_Mean                  0.6771
avg_Recall_Seed_Std                   0.0944
avg_Recall_Seed_Min                     0.48
avg_Recall_Seed_Max                     0.92
avg_NPV_Seed_Mean                     0.7186
avg_NPV_Seed_Std                      0.0579
avg_NPV_Seed_Min                      0.5517
avg_NPV_Seed_Max                      0.8947
avg_Specificity_Seed_Mean             0.7889
avg_Specificity_Seed_Std              0.0914
avg_Specificity_Seed_Min                 0.5
avg_Specif

In [ ]:
hidden_size_smaller = range(features_size//3, features_size*2)
lambda_value_wider = 2.0 ** np.arange(-20, 21)

optimization_record,optimization_result = evaluator.grid_search_hidden_size_and_lambda(
    hidden_size_range=hidden_size_smaller,
    lambda_range=lambda_value_wider
)

In [ ]:
best_optimization_result = extract_optimal_vector(optimization_result)
best_optimization_result

In [ ]:
model_configs = [
    {
        "Hidden_Nodes": best_hidden_size['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_hidden_size['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_lambda_value['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_lambda_value['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_combination_result['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_combination_result['Lambda_Value']
    },
    {
        "Hidden_Nodes": best_optimization_result['Hidden_Nodes'],
        "Activation": sigmoid,
        "Lambda_Value": best_optimization_result['Lambda_Value']
    }
]

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

x_test = x_test.reset_index(drop=True)
y_test = y_test.reset_index(drop=True)
x_train = x_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)


main_scaler = StandardScaler()
x_train_scaled = pd.DataFrame(main_scaler.fit_transform(x_train), columns=X.columns)
x_test_scaled  = pd.DataFrame(main_scaler.transform(x_test), columns=X.columns)

In [ ]:
testing_results = []
for config in model_configs:
    print(f"Executing: Nodes : {config['Hidden_Nodes']} , lambda value : {config['Lambda_Value']} , activation function : {config['Activation'].__name__}")

    elm = ExtremeLearningMachine(
        features_size=features_size,
        hidden_size=config["Hidden_Nodes"],
        activation_function=config["Activation"],
        regularization_lambda=config["Lambda_Value"]
    )
    elm.initialize_random_weights(random_seed=42)


    elm.fit(x_train_scaled.values, y_train.values)
    y_pred = elm.predict(x_test_scaled.values)

    evaluation = EvaluationMatrix(y_test, y_pred)
    print(evaluation.get_report())

    metrics = evaluation.get_all_metrics()
    test_result = {
        "Hidden_Nodes" : config['Hidden_Nodes'],
        "Lambda_Value" : config['Lambda_Value'],
        "Activation"   : config['Activation'].__name__
    }
    test_result.update(metrics)
    testing_results.append(test_result)
    print("\n")

In [ ]:
final_model_result = pd.DataFrame(testing_results)
final_model_result